In [1]:
from acled_model import (
    pre_process_data,
    train_evaluate_model,
    split_data,
    calculate_conflict_ratio,
    grouped_timeseries_cv_ids,
    verify_cv_splits,
    timeseries_cross_val_predict,
)
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
)

INFO:ingest.acled:Access token correctly retrieved.
INFO:acled_model:Data grouped by sub_event_type
INFO:acled_model:Escalation target set at 0.5 standard deviations above the mean.


In [2]:
all_data = pd.read_csv("../data/all_data.csv")

countries = ["Sudan"]
start_date = "2017-07-01"  # TODO validation for 6 month warm up period
end_date = "2024-12-31"

train_start_date = "2018-01-01"
train_end_date = "2022-12-31"

onset_start_date = "2023-01-01"
onset_end_date = "2023-12-31"

active_start_date = "2024-01-01"
active_end_date = "2024-12-31"

## Testing to see which k deviations from the mean works best

In [7]:
test_ks = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
results = []

for k in test_ks:
    result = train_evaluate_model(all_data, k, "event_type")
    result["k"] = k
    results.append(result)

print(results)

INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 0.25 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 0.5 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 0.75 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 1.0 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 1.25 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 1.5 standard deviations above the mean.


[{'optimal_threshold': '0.0345', 'onset_aupr': '0.4706', 'onset_precision_class1': '0.4229', 'onset_recall_class1': '1.0000', 'onset_f1_class1': '0.5944', 'active_aupr': '0.4715', 'active_precision_class1': '0.4318', 'active_recall_class1': '0.9896', 'active_f1_class1': '0.6013', 'k': 0.25}, {'optimal_threshold': '0.0465', 'onset_aupr': '0.3817', 'onset_precision_class1': '0.3795', 'onset_recall_class1': '1.0000', 'onset_f1_class1': '0.5502', 'active_aupr': '0.4096', 'active_precision_class1': '0.4019', 'active_recall_class1': '1.0000', 'active_f1_class1': '0.5733', 'k': 0.5}, {'optimal_threshold': '0.0249', 'onset_aupr': '0.4185', 'onset_precision_class1': '0.3679', 'onset_recall_class1': '0.9873', 'onset_f1_class1': '0.5361', 'active_aupr': '0.3620', 'active_precision_class1': '0.3382', 'active_recall_class1': '0.9583', 'active_f1_class1': '0.5000', 'k': 0.75}, {'optimal_threshold': '0.0007', 'onset_aupr': '0.3584', 'onset_precision_class1': '0.3167', 'onset_recall_class1': '1.0000',

## Testing to see whether sub events or events work better
I think that sub-events might be too sparse so event_type might be better

In [5]:
result = train_evaluate_model(all_data, 0.5)
print("sub_event_type", result)

result = train_evaluate_model(all_data, 0.5, "event_type")
print("event_type", result)

INFO:acled_model:Data grouped by sub_event_type
INFO:acled_model:Escalation target set at 0.5 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 0.5 standard deviations above the mean.


sub_event_type {'optimal_threshold': '0.0880', 'onset_aupr': '0.3847', 'onset_precision_class1': '0.3825', 'onset_recall_class1': '0.9765', 'onset_f1_class1': '0.5497', 'active_aupr': '0.3989', 'active_precision_class1': '0.3991', 'active_recall_class1': '0.9884', 'active_f1_class1': '0.5686'}
event_type {'optimal_threshold': '0.0465', 'onset_aupr': '0.3817', 'onset_precision_class1': '0.3795', 'onset_recall_class1': '1.0000', 'onset_f1_class1': '0.5502', 'active_aupr': '0.4096', 'active_precision_class1': '0.4019', 'active_recall_class1': '1.0000', 'active_f1_class1': '0.5733'}


## Testing number of cv splits

In [ ]:
results = train_evaluate_model(all_data, k=0.5, event_col="event_type", n_splits=4)
print("Four splits in cv", results)

results = train_evaluate_model(all_data, k=0.5, event_col="event_type", n_splits=5)

## Testing different params

In [ ]:
def train_evaluate_model(all_data, params):
    # Process data
    processed_df, predictor_cols = pre_process_data(
        all_data, params["k"], params["event_col"]
    )

    # Split data
    train_df, y_train, X_train = split_data(
        processed_df,
        predictor_cols,
        train_start_date,
        train_end_date,
    )
    onset_df, y_onset, X_onset = split_data(
        processed_df,
        predictor_cols,
        onset_start_date,
        onset_end_date,
    )
    active_df, y_active, X_active = split_data(
        processed_df,
        predictor_cols,
        active_start_date,
        active_end_date,
    )

    ratios = calculate_conflict_ratio(train_df)

    scale_weight = ratios["non-escalation"] / ratios["escalation"]
    xgb_model = xgb.XGBClassifier(
        scale_pos_weight=scale_weight,
        eval_metric="aucpr",  # As decided in proposal
        random_state=7,
    )

    grouped_timeseries_cv = list(
        grouped_timeseries_cv_ids(train_df["year_month"], n_splits=params["n_splits"])
    )

    verify_cv_splits(train_df, grouped_timeseries_cv)

    param_grid = params.copy()
    del param_grid["k"]
    del param_grid["event_col"]
    del param_grid["n_splits"]

    # 2. Replace GridSearchCV with RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=xgb_model,
        param_distributions=param_grid,
        n_iter=150,  # The number of models to test (adjust based on your time)
        cv=grouped_timeseries_cv,
        scoring="average_precision",
        n_jobs=-1,
        random_state=23,
    )

    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_
    best_params = random_search.best_params_

    # Take number of true y and predicted y for training data out of fold sample
    oof_y_true, oof_y_proba = timeseries_cross_val_predict(
        best_model, X_train, y_train, grouped_timeseries_cv
    )

    # Tune threshold on onset (validation/test partition)
    # y_pred_proba_onset = best_model.predict_proba(X_onset)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(oof_y_true, oof_y_proba)
    f1_scores = (2 * precisions * recalls / (precisions + recalls + 1e-10))[:-1]
    optimal_threshold = thresholds[np.argmax(f1_scores)]

    # Evaluate on onset test set
    y_pred_proba_onset = best_model.predict_proba(X_onset)[:, 1]
    y_pred_custom_onset = (y_pred_proba_onset >= optimal_threshold).astype(int)

    # Evaluate on active test set
    y_pred_proba_active = best_model.predict_proba(X_active)[:, 1]
    y_pred_custom_active = (y_pred_proba_active >= optimal_threshold).astype(int)

    onset_report = classification_report(
        y_onset, y_pred_custom_onset, output_dict=True, zero_division=0
    )
    active_report = classification_report(
        y_active, y_pred_custom_active, output_dict=True, zero_division=0
    )

    class_key = "1" if "1" in onset_report else 1

    results = {
        "optimal_threshold": f"{optimal_threshold:.4f}",
        # Onset Metrics
        "onset_aupr": f"{average_precision_score(y_onset, y_pred_proba_onset):.4f}",
        "onset_precision_class1": f"{onset_report[class_key]['precision']:.4f}",
        "onset_recall_class1": f"{onset_report[class_key]['recall']:.4f}",
        "onset_f1_class1": f"{onset_report[class_key]['f1-score']:.4f}",
        # Active Metrics
        "active_aupr": f"{average_precision_score(y_active, y_pred_proba_active):.4f}",
        "active_precision_class1": f"{active_report[class_key]['precision']:.4f}",
        "active_recall_class1": f"{active_report[class_key]['recall']:.4f}",
        "active_f1_class1": f"{active_report[class_key]['f1-score']:.4f}",
    }
    return results, best_params

In [ ]:
all_params = {
    # Existing
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "max_delta_step": [0, 1, 5],
    "gamma": [0, 1, 3, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 2],
    "reg_lambda": [1, 5, 10],
    "colsample_bylevel": [0.6, 0.8, 1.0],
    "k": 0.75,
    "event_col": "event_type",
    "n_splits": 4,
}

In [ ]:
results, best_params = train_evaluate_model(all_data, all_params)